在上一章中，我们采用了不同于学过的数值微分法来对误差函数对权重参数的梯度，这种方法叫做误差反向传播法，虽然比数值微分法复杂，但其计算梯度的速度远远高于数值微分法，可以达到几千上万倍，所以现在学习一下这个方法

### 1.原理

- 工具

计算表，将参与的要素从左到右依次排开，并且用神经元的形式表示它们之间的算数关系，文字不便描述，看书即可

- 数学基础

链式法则：在计算表中，下一个神经元对应之前的神经元有一个变化率(即导数)，是指上一个元素略微改变对下一个元素大小改变的程度，在计算表中，往往不止一次计算，在正向传播的过程中，记录下这些导数，实现反向传播的过程就是从一个“1”的导数为止，不断乘以前面记录下的导数，从而找到最终结果和最初元素之间的导数

- 分类

1. 加法节点：如我们所知，当两个节点以加减法形式组成下一个节点时，下一个节点对上两个节点的导数分别为其系数，所以反向传播时，只用乘回去即可
2. 乘法节点：以z=xy举例，对x求偏导等于y，对y求偏导等于x，我们可知，要乘以对方在正向传播时的导数

### 2.简单层的实现

把每一个运算定义为一个层的话，我们现在可以实现关于乘法的反向传播层

In [ ]:
from numba.core.types import none

from 神经网络的学习 import cross_entropy_error


class MulLayer:
    def __init__(self):#init函数创建两个值，用来保存输入值
        self.x=none
        self.y=none
    def forward(self,x,y   ):#forward表示正向传播
        self.x=x
        self.y=y
        out=x*y
        return out
    def backward(self,dout):#dout表示的是从上游传来的导数值
        dx=dout*self.y
        dy=dout*self.x
        return dx,dy

现在我们可以举个例子，一个苹果100元，买了两个，税是0.1：

In [4]:
from numba.core.types import none


class MulLayer:
    def __init__(self):#init函数创建两个值，用来保存输入值
        self.x=None
        self.y=None
    def forward(self,x,y   ):#forward表示正向传播
        self.x=x
        self.y=y
        out=x*y
        return out
    def backward(self,dout):#dout表示的是从上游传来的导数值
        dx=dout*self.y
        dy=dout*self.x
        return dx,dy
apple=100
apple_num=2
tax=1.1
#layer
mul_apple_layer=MulLayer()
mul_tax_layer=MulLayer()
#forward
apple_price=mul_apple_layer.forward(apple,apple_num)
price=mul_tax_layer.forward(apple_price,tax)
print(price)
#backward
dprice=1
dapple_price,dtax=mul_tax_layer.backward(dprice)
dapple,dapple_num=mul_apple_layer.backward(dapple_price)
print(dapple,dapple_num,dtax)


220.00000000000003
2.2 110.00000000000001 200


上面就是典型的一个简单乘法层正反向传播的例子，结果是正确的，下面是一个简单的加法层

In [ ]:
class AddLayer:
    def __init__(self):
        pass
    def forward(self,x,y ):
        out=x+y
        return out
    def backward(self,dout):
        dx=dout*1
        dy=dout*1
        return dx,dy
apple = 100
apple_num = 2
orange = 150
orange_num = 3
tax = 1.1

# layer
mul_apple_layer = MulLayer()
mul_orange_layer = MulLayer()
add_apple_orange_layer = AddLayer()
mul_tax_layer = MulLayer()

# forward
apple_price = mul_apple_layer.forward(apple, apple_num)  # (1)
orange_price = mul_orange_layer.forward(orange, orange_num)  # (2)
all_price = add_apple_orange_layer.forward(apple_price, orange_price)  # (3)
price = mul_tax_layer.forward(all_price, tax)  # (4)

# backward
dprice = 1
dall_price, dtax = mul_tax_layer.backward(dprice)  # (4)
dapple_price, dorange_price = add_apple_orange_layer.backward(dall_price)  # (3)
dorange, dorange_num = mul_orange_layer.backward(dorange_price)  # (2)
dapple, dapple_num = mul_apple_layer.backward(dapple_price)  # (1)

print("price:", int(price))
print("dApple:", dapple)
print("dApple_num:", int(dapple_num))
print("dOrange:", dorange)
print("dOrange_num:", int(dorange_num))
print("dTax:", dtax)


### 3.激活函数层的实现

1.Relu层的实现：

回忆一下ReLU函数是将比零大的数返回其本身，而小于等于0的数全部返回0，根据原理我们可以写出如下代码：

In [ ]:
class ReLU:
    def __init__(self):
        self.mask=None#告诉编译器，这里有一个mask占位了
    def forward(self,x):
        self.mask=(x<=0)#经过这一步，得到了一个与x形状相同的矩阵，不过里面的大于零的部分全部变成了False，剩余的为True，
        out=x.copy()#copy一个x矩阵
        out[self.mask]=0#这里实际上是两个矩阵，运用到布尔索引，编译器找到mask中为ture的地方，并把out中相同位置的元素全部替换成零
        return out
    def backward(self   ,dout):
        dout[self.mask]=0#同理，为零的导数也为零，否则就是不变
        dx=dout
        return dx


### 3.sigmoid函数层的实现

$$
\sigma(x) = \frac{1}{1 + e^{-x}}
$$

由于复合函数较为复杂，在方向传播过程中可分为四步，假设1/x=y

1. 计算1/x,其导数为-1/x²，所以第一步dout*y²
2. 加1，dout不变
3. exp(-x),所以dout再乘以exp(-x)
4. x变为-x，再乘-1

将上述四步化简我们可以得到最终答案dout=dout(原)×y²乘exp(-x)，即是dout(原)*y*(1-y),也就是说我们只需要根据正向输出的结果就能计算出反向传播。代码实现如下

In [ ]:
import numpy as np
class sigmoid:
    def __init__(self):
        self.out=None
    def forward(self,x):
        out=1/(1+np.exp(-x))
        self.out=out
        return out
    def backward(self,dout):
        dout=dout*(1.0-self.out)*self.out

### 4.Affine/softmax层的实现

1. Affine层

我们知道在计算加权信号总和的时候，我们用到Y=np.dot(w,x)+B，然后经过激活函数转换后传递给下一层，在线性代数里，把这个变换叫做放射变化，包含一次线性变换和一次平移，对应的是神经网络李安的加权和运算与加偏置运算，而在神经网络里面，实现这一功能的就叫做Affine层，Affine层里面的计算不同于先前的计算，先前在网络间流动的都是标量，而现在是numpy数组！

假设前向传播为 $Y = XW + b$，反向传播的梯度公式如下：

1. **输入的梯度**：
$$ \frac{\partial L}{\partial X} = \frac{\partial L}{\partial Y} W^T $$

2. **权重的梯度**：
$$ \frac{\partial L}{\partial W} = X^T \frac{\partial L}{\partial Y} $$

3. **偏置的梯度**：
$$ \frac{\partial L}{\partial b} = \sum_{axis=0} \frac{\partial L}{\partial Y} $$

其中的T表示的是转置，上述公式的推到可以基于一致性推出，因为进行乘法运算的矩阵要求对应维度的元素个数保持一致，下面是代码实现Affine层(当处理数据个数不为一时)

In [ ]:
class Affine:
    def __init__(self,w,b):
        self.w=w
        self.b=b
        self.x=None
        self.dw=None
        self.db=None
    def forward(self,x):
        self.x=x
        out=np.dot(x,self.w)+self.b
        return out
    def backward(self,dout):
        dx=np.dot(dout,self.w.T)
        self.dw=np.dot(self.x.T,dout)
        self.db=np.sum(dout,axis=0)
        return dx


2. softmax-with-loss层

神经网络中进行的处理有推理和学习两个阶段，推理阶段一般不用softmax函数来进行正规化，这样的结果一般被称为得分，当神经网络的推理只需要给出一个结果时，因为只对最大值感兴趣所以不需要softmax函数

但当softmax函数出现时，我们往往也会配合使用交叉熵误差函数来判断学习好坏，所以这里一并实现了

这里看图更加清晰(p152)，用文字简单描述就是softmax函数完成正规化传入交叉熵函数层，那么在反向传播时，经过交叉熵层后输出的是(y1-t1,y2-t2,y3-t3),y1,y2,y3都是softmax函数的输出，而t1,t2,t3是监督数据的标签，反向传播的结果就是softmax的输出和监督数据的差分，然后将其传递给前面的层。当识别误差大时，softmax前面的层会学到“大”内容，反之学得很少。下面是代码实现：

In [ ]:
class SoftmaxWithLoss:
    def __init__(self):
        self.loss=None#损失
        self.y=None#softmax的输出
        self.t=None#监督数据
    def forward(self,x,t):
        self.t=t
        self.y=softmax(x)
        self.loss=cross_entropy_error(self.y,self.t)
        return self.loss
    def backward(self,dout=1):#因为交叉熵是在最后一步，所以这里相当于反向传播的起点，dout自然设为0
        batch_size=self.t.shape[0]
        dx=(self.y-self.t)/batch_size
        return dx



### 5.误差反向传播法的实现！

已经学习了各种层，现在正式构建神经网络！先构造整体流程：

1. 从训练数据中随机选择一部分数据(mini-batch)
2. 计算损失函数关于各个权重参数的梯度(用到这章的误差反向传播法)
3. 将权重参数沿梯度方向更新
4. 重复1，2，3步直到误差函数足够小

下面就完整的写一写这个流程！

In [ ]:
import numpy as np
from common.layers import *
from common.gradient import numerical_gradient
from collections import OrderedDict
#引入库
class TwoLawyerNet:
    #初始化权重
    def __init__(self,input_size,output_size,hidden_size,weight_init_std=0.01):
        self_params={}
        self_params['W1']=weight_init_std*np.random.randn(input_size,hidden_size)
        self_params['b1']=np.zeros(hidden_size)
        self_params['W2']=weight_init_std*np.random.randn(hidden_size,output_size)
        self_params['b2']=np.zeros(output_size)
        #生成层
        self.layers=OrderedDict#OrderdDict是一个创造有序字典的方法，把我们松散的各种层链接在一起,字典的顺序，就是你创造的顺序，因此只需要调用各层的forward方法就可以推进神经网络
        self.layers['Affine1']=Affine(self.paras['W1'],self.params['b1'])#第一个，仿射变换层
        self.layers['Relu1']=Relu()#跟在Affine1后做变换
        self.layers['Affine2']=Affine(self.paras['W2'],self.params['b2'])
        self.lastLayer=SoftmaxWithLoss()

    def predict(self,x):#推动神经网络正向传播的方法
        for layer in self.layers.values():
            x=layer.forward(x)
            return x

    def loss(self,x,t):
        y=self.predict(x)
        return self.lastLayer.forward(y,t)
    def accuracy(self,x,t):
        y=self.predict(x)
        y=np.argmax(y,axis=1)
        if t.ndim!=1:
            t=np.argmax(t,axis=1)
        accuracy=np.sum(y==t)/float(x.shape[0])
        return accuracy
    def numerical_gradient(self,x,t):
        loss_W=lambda W:self.loss(x,t)
        grads={}
        grads['W1']=numerical_gradient(loss_W,self.param['W1'])
        grads['b1']=numerical_gradient((loss_W,self.param['b1']))
        grads['W2']=numerical_gradient(loss_W,self.param['W2'])
        grads['b2']=numerical_gradient((loss_W,self.param['b2']))
        return grads
    def gradient(self,x,t):#重中之重！
        #forward
        self.loss(x,t)#先正向遍历所有层算出损失函数
        #backward
        dout=1
        dout=self.lastLayer.backward(dout)#先从后面反一层回去
        layers=list(self.layers.values())#取出所有层变成真正的列表，便于用列表方法倒置
        layers.reverse()#列表导致，然后从后面往前面遍历
        for layer in layers:
            dout=layer.backward(dout)

        #设定

        grads={}
        grads['W1']=self.layers['Affine1'].dw
        grads['b1']=self.layers['Affine1'].db
        grads['W2']=self.layers['Affine2'].dw
        grads['b2']=self.layers['Affine2'].db
        return grads

上述代码非常具有工程思维，利用层构造起神经网络，这样以后构造更多更复杂的神经网络时，可以像拼积木一样将不同的层拼起来

### 6.一些附加

利用反向传播法算梯度非常的快速，数值微分法用的很少了，一般只用来验证反向传播法是否正确，这叫做梯度确认顺便，附加完整版代码

In [ ]:

import sys, os
sys.path.append(os.pardir)

import numpy as np
from mnist import load_mnist
from two_layer_net import TwoLayerNet


(x_train, t_train), (x_test, t_test) = load_mnist(normalize=True, one_hot_label=True)
import numpy as np
from common.layers import *
from common.gradient import numerical_gradient
from collections import OrderedDict
#引入库
class TwoLawyerNet:
    #初始化权重
    def __init__(self,input_size,output_size,hidden_size,weight_init_std=0.01):
        self_params={}
        self_params['W1']=weight_init_std*np.random.randn(input_size,hidden_size)
        self_params['b1']=np.zeros(hidden_size)
        self_params['W2']=weight_init_std*np.random.randn(hidden_size,output_size)
        self_params['b2']=np.zeros(output_size)
        #生成层
        self.layers=OrderedDict#OrderdDict是一个创造有序字典的方法，把我们松散的各种层链接在一起,字典的顺序，就是你创造的顺序，因此只需要调用各层的forward方法就可以推进神经网络
        self.layers['Affine1']=Affine(self.paras['W1'],self.params['b1'])#第一个，仿射变换层
        self.layers['Relu1']=Relu()#跟在Affine1后做变换
        self.layers['Affine2']=Affine(self.paras['W2'],self.params['b2'])
        self.lastLayer=SoftmaxWithLoss()

    def predict(self,x):#推动神经网络正向传播的方法
        for layer in self.layers.values():
            x=layer.forward(x)
            return x

    def loss(self,x,t):
        y=self.predict(x)
        return self.lastLayer.forward(y,t)
    def accuracy(self,x,t):
        y=self.predict(x)
        y=np.argmax(y,axis=1)
        if t.ndim!=1:
            t=np.argmax(t,axis=1)
        accuracy=np.sum(y==t)/float(x.shape[0])
        return accuracy
    def numerical_gradient(self,x,t):
        loss_W=lambda W:self.loss(x,t)
        grads={}
        grads['W1']=numerical_gradient(loss_W,self.param['W1'])
        grads['b1']=numerical_gradient((loss_W,self.param['b1']))
        grads['W2']=numerical_gradient(loss_W,self.param['W2'])
        grads['b2']=numerical_gradient((loss_W,self.param['b2']))
        return grads
    def gradient(self,x,t):#重中之重！
        #forward
        self.loss(x,t)#先正向遍历所有层算出损失函数
        #backward
        dout=1
        dout=self.lastLayer.backward(dout)#先从后面反一层回去
        layers=list(self.layers.values())#取出所有层变成真正的列表，便于用列表方法倒置
        layers.reverse()#列表导致，然后从后面往前面遍历
        for layer in layers:
            dout=layer.backward(dout)

        #设定

        grads={}
        grads['W1']=self.layers['Affine1'].dw
        grads['b1']=self.layers['Affine1'].db
        grads['W2']=self.layers['Affine2'].dw
        grads['b2']=self.layers['Affine2'].db
        return grads



#建立网络
network = TwoLayerNet(input_size=784, hidden_size=50, output_size=10)
#检查机制，防止过拟合
iters_num = 10000
train_size = x_train.shape[0]
batch_size = 100
learning_rate = 0.1

train_loss_list = []
train_acc_list = []
test_acc_list = []

iter_per_epoch = max(train_size / batch_size, 1)
#更新参数
for i in range(iters_num):
    batch_mask = np.random.choice(train_size, batch_size)
    x_batch = x_train[batch_mask]
    t_batch = t_train[batch_mask]


    grad = network.gradient(x_batch, t_batch)

    # 更新
    for key in ('W1', 'b1', 'W2', 'b2'):
        network.params[key] -= learning_rate * grad[key]

    loss = network.loss(x_batch, t_batch)
    train_loss_list.append(loss)

    if i % iter_per_epoch == 0:
        train_acc = network.accuracy(x_train, t_train)
        test_acc = network.accuracy(x_test, t_test)
        train_acc_list.append(train_acc)
        test_acc_list.append(test_acc)
        print(train_acc, test_acc)
